# 01 Data Ingestion and Cleaning

This notebook loads the core Olist datasets, performs inital data inspection and cleaning, joins the required tables, and produces a shared processed dataset for the rest of the project.

**Output:** `../data/processed/orders_core.parquet`

## 1. Environment Check

This section confirms that the notebook is using the correct Python environment.

In [1]:
import sys
print(sys.executable)

c:\Users\John\AppData\Local\Programs\Python\Python314\python.exe


## 2. Start Spark Session

Initialize the Spark session used for data ingestion and preprocessing.

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Olist_Ingestion").getOrCreate()

## 3. Load Core Datasets

Load the main Olist CSV files required for the project:
- orders
- customers
- order items
- products
- payments

In [3]:
orders = spark.read.csv("../data/raw/olist_orders_dataset.csv", header=True, inferSchema = True)
customers = spark.read.csv("../data/raw/olist_customers_dataset.csv", header=True, inferSchema=True)
order_items = spark.read.csv("../data/raw/olist_order_items_dataset.csv", header=True, inferSchema=True)
products = spark.read.csv("../data/raw/olist_products_dataset.csv", header=True, inferSchema=True)
payments = spark.read.csv("../data/raw/olist_order_payments_dataset.csv", header=True, inferSchema=True)

categories = spark.read.csv("../data/raw/product_category_name_translation.csv", header=True, inferSchema=True)
geolocations = spark.read.csv("../data/raw/olist_geolocation_dataset.csv", header=True, inferSchema=True)
reviews = spark.read.csv("../data/raw/olist_order_reviews_dataset.csv", header=True, inferSchema=True)
sellers = spark.read.csv("../data/raw/olist_sellers_dataset.csv", header=True, inferSchema=True)

In [4]:
print("orders:", orders.count())
print("customers:", customers.count())
print("order_items:", order_items.count())
print("products:", products.count())
print("payments:", payments.count())

orders: 99441
customers: 99441
order_items: 112650
products: 32951
payments: 103886


## 4. Inspect Schemas and Sample Rows

Preview the structure and example rows of each dataset to understand available columns and data types.

In [5]:
orders.printSchema()
orders.show(5)
order_items.printSchema()
order_items.show(5)
customers.printSchema()
customers.show(5)
products.printSchema()
products.show(5)
payments.printSchema()
payments.show(5)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------

## 5. Check Missing Values

Measure missing values in the main datasets to identify columns that may need cleaning or special handling.

In [6]:
from pyspark.sql.functions import col, count, when

orders.select([count(when(col(c).isNull(), c)).alias(c) for c in orders.columns]).show()
customers.select([count(when(col(c).isNull(), c)).alias(c) for c in customers.columns]).show()
products.select([count(when(col(c).isNull(), c)).alias(c) for c in products.columns]).show()
payments.select([count(when(col(c).isNull(), c)).alias(c) for c in payments.columns]).show()
order_items.select([count(when(col(c).isNull(), c)).alias(c) for c in order_items.columns]).show()

+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|       0|          0|           0|                       0|              160|                        1783|                         2965|                            0|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+

+-----------+------------------+------------------------+-------------+--------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|c

## 6. Select Core Columns

Keep only the columns needed for the project to simplify the joined dataset and reduce unnecessary complexity.

In [8]:
orders_sel = orders.select(
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp"
)

customers_sel = customers.select(
    "customer_id",
    "customer_unique_id",
    "customer_city",
    "customer_state"
)

order_items_sel = order_items.select(
    "order_id",
    "order_item_id",
    "product_id",
    "seller_id",
    "price",
    "freight_value"
)

products_sel = products.select(
    "product_id",
    "product_category_name"
)

orders_sel.printSchema()
order_items_sel.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)



## 7. Aggregate Payment Data

Aggregate payments at the order level before joining, so that payment values are not duplicated in later joins.

In [9]:
from pyspark.sql.functions import sum

payments_agg = payments.groupBy("order_id").agg(
    sum("payment_value").alias("payment_value_total")
)

payments_agg.show(5)
print("payments_agg:", payments_agg.count())

+--------------------+-------------------+
|            order_id|payment_value_total|
+--------------------+-------------------+
|bb2d7e3141540afc2...|              37.15|
|85be7c94bcd3f908f...|              72.75|
|8ca5bdac5ebe8f2d6...| 189.07999999999998|
|54066aeaaf3ac32e7...|             148.06|
|5db54d41d5ebd6d76...|             136.26|
+--------------------+-------------------+
only showing top 5 rows
payments_agg: 99440


## 8. Join Datasets into a Core Table

Join the cleaned and selected datasets into a single shared DataFrame called `orders_core`.

In [10]:
orders_core = (
    orders_sel
    .join(customers_sel, on="customer_id", how="left")
    .join(order_items_sel, on="order_id", how="left")
    .join(products_sel, on="product_id", how="left")
    .join(payments_agg, on="order_id", how="left")
)

**Note:** After joining with `order_items`, the resulting dataset is at **order-item level** rather than strictly one row per order. This means one order may appear multiple times if it contains multiple items.

## 9. Create Derived Columns

Create additional columns used by the project, such as:
- year
- month
- total order value
- fallback category name for missing values

In [11]:
from pyspark.sql.functions import year, month, col, when

orders_core = (
    orders_core
    .withColumn("year", year("order_purchase_timestamp"))
    .withColumn("month", month("order_purchase_timestamp"))
    .withColumn("total_order_value", col("price") + col("freight_value"))
    .withColumn("product_category_name", when(col("product_category_name").isNull(), "unknown").otherwise(col("product_category_name")))
)

## 10. Validate Final Dataset

Inspect the schema, preview the rows, and count the total number of records in the final joined dataset.

In [12]:
orders_core.printSchema()
orders_core.show(10, truncate=False)
print("orders_core row count:", orders_core.count())

root
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- payment_value_total: double (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- total_order_value: double (nullable = true)

+--------------------------------+--------------------------------+--------------------------------+------------+------------------------+--------------------------------+-----------------------+--------------+-------------+----

## 11. Check Missing Values in Final Output

Confirm the quality of the joined dataset and identify any remaining missing values.

In [13]:
from pyspark.sql.functions import count, when

orders_core.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in orders_core.columns
]).show()

+--------+----------+-----------+------------+------------------------+------------------+-------------+--------------+-------------+---------+-----+-------------+---------------------+-------------------+----+-----+-----------------+
|order_id|product_id|customer_id|order_status|order_purchase_timestamp|customer_unique_id|customer_city|customer_state|order_item_id|seller_id|price|freight_value|product_category_name|payment_value_total|year|month|total_order_value|
+--------+----------+-----------+------------+------------------------+------------------+-------------+--------------+-------------+---------+-----+-------------+---------------------+-------------------+----+-----+-----------------+
|       0|       775|          0|           0|                       0|                 0|            0|             0|          775|      775|  775|          775|                    0|                  3|   0|    0|              775|
+--------+----------+-----------+------------+--------------

## 12. Save Processed Dataset

Write the final processed dataset to Parquet format for use by the rest of the team.

In [14]:
orders_core.write.mode("overwrite").parquet("../data/processed/orders_core.parquet")

Py4JJavaError: An error occurred while calling o629.parquet.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset. -see https://cwiki.apache.org/confluence/display/HADOOP2/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:789)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:298)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:314)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:1179)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:861)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:901)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:873)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:900)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdir

## 13. Verify Saved Output

Reload the saved Parquet file and preview it to confirm that the write operation succeeded.

In [31]:
orders_core_check = spark.read.parquet("../data/processed/orders_core.parquet")
orders_core_check.printSchema()
orders_core_check.show(10, truncate=False)
print("saved orders_core row count:", orders_core_check.count())

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/c:/Users/John/Desktop/data_sience_group/BigData-Ecommerce-Analysis/data/processed/orders_core.parquet. SQLSTATE: 42K03